# Unified validation notebook (Artem vs David)

Imports strategy code from airlock modules, runs time-split holdout + walk-forward, and does baselines + swapped-parameter experiments.
Kernel-stability fixes: cache Bar objects; disable equity curves during grid search; avoid QC .NET Bar collisions by using entry_exit.Bar explicitly.


In [ ]:
# region imports
from AlgorithmImports import *
from datetime import datetime, timedelta
import numpy as np
import pandas as pd
# endregion

# region project imports (as requested)
from entry_exit import *
from stop_loss import *
from swing_high_low_detection import *
from risk import *
# endregion

# IMPORTANT: keep a module handle to avoid name collisions with QC (.NET Bar, etc.)
import entry_exit as ee

print("entry_exit:", ee.__file__)
import stop_loss as sl_mod, swing_high_low_detection as sw_mod, risk as rk_mod
print("stop_loss:", sl_mod.__file__)
print("swing_high_low_detection:", sw_mod.__file__)
print("risk:", rk_mod.__file__)

In [2]:
# --- Data helpers (QC QuantBook)

qb = QuantBook()

def load_crypto_history(symbol: str, start: datetime, end: datetime, resolution=Resolution.Minute) -> pd.DataFrame:
    """Return OHLCV dataframe indexed by datetime (QuantConnect Research)."""
    s = qb.AddCrypto(symbol, resolution).Symbol
    hist = qb.History(s, start, end, resolution)

    if hist is None or len(hist) == 0:
        raise ValueError(f"No history for {symbol} in [{start}..{end}] at {resolution}")

    # QC History sometimes returns a MultiIndex with (symbol,time) or (time,symbol)
    if isinstance(hist.index, pd.MultiIndex):
        # Try to slice by Symbol object first, but detect the correct level
        key_candidates = [s, str(s), symbol]
        try:
            key_candidates.append(s.Value)
        except Exception:
            pass

        found = False
        for lvl in range(hist.index.nlevels):
            lvl_vals = hist.index.get_level_values(lvl)

            # Compare both raw objects (for Symbol) and strings (for safety)
            for k in key_candidates:
                try:
                    if (lvl_vals == k).any():
                        hist = hist.xs(k, level=lvl, drop_level=True)
                        found = True
                        break
                except Exception:
                    pass

                # string fallback
                try:
                    lvl_str = lvl_vals.astype(str)
                    if (lvl_str == str(k)).any():
                        hist = hist.xs(k, level=lvl, drop_level=True)
                        found = True
                        break
                except Exception:
                    pass

            if found:
                break

        if not found:
            # helpful debug
            names = list(hist.index.names)
            samples = {}
            for i in range(hist.index.nlevels):
                u = hist.index.get_level_values(i).unique()[:5]
                samples[f"level_{i}_name"] = names[i] if i < len(names) else None
                samples[f"level_{i}_sample"] = list(u.astype(str))
            raise ValueError(f"Couldn't slice history by symbol={symbol}. MultiIndex info: {samples}")

    # Normalize column names
    cols = {c.lower(): c for c in hist.columns}

    def getcol(name: str):
        # accept both exact and case-insensitive matches
        if name in hist.columns:
            return hist[name]
        key = name.lower()
        if key in cols:
            return hist[cols[key]]
        raise KeyError(f"Missing column {name} in history for {symbol}. Columns: {hist.columns.tolist()}")

    out = pd.DataFrame({
        "open": getcol("open").astype(float),
        "high": getcol("high").astype(float),
        "low":  getcol("low").astype(float),
        "close":getcol("close").astype(float),
    })

    # Volume may be missing depending on source
    if "volume" in cols or "volume" in hist.columns:
        out["volume"] = getcol("volume").astype(float)
    else:
        out["volume"] = 0.0

    out.index = pd.to_datetime(out.index)
    return out.sort_index()


def resample_ohlc(df: pd.DataFrame, rule: str = "15min") -> pd.DataFrame:
    # use 'min' not 'T' to avoid pandas FutureWarning
    o = df["open"].resample(rule).first()
    h = df["high"].resample(rule).max()
    l = df["low"].resample(rule).min()
    c = df["close"].resample(rule).last()
    v = df["volume"].resample(rule).sum() if "volume" in df.columns else None
    out = pd.concat([o, h, l, c, v], axis=1).dropna()
    out.columns = ["open", "high", "low", "close", "volume"]
    return out

def compute_swings(df_ohlc: pd.DataFrame) -> pd.DataFrame:
    """Run swing_highs_lows_online and align result to df_ohlc index."""
    sw = swing_highs_lows_online(df_ohlc)
    if sw is None or len(sw) == 0:
        return pd.DataFrame(index=df_ohlc.index, data={"HighLow": np.nan, "Level": np.nan})

    out = sw
    if not out.index.equals(df_ohlc.index):
        out = out.reindex(df_ohlc.index)

    if "HighLow" not in out.columns:
        for alt in ["highlow", "HighLowFlag", "flag"]:
            if alt in out.columns:
                out["HighLow"] = out[alt]
                break
    if "Level" not in out.columns:
        for alt in ["level", "Price", "price"]:
            if alt in out.columns:
                out["Level"] = out[alt]
                break

    if "HighLow" not in out.columns:
        out["HighLow"] = np.nan
    if "Level" not in out.columns:
        out["Level"] = np.nan

    return out[["HighLow", "Level"]]

In [3]:
# --- Configs (Artem vs David)

from dataclasses import dataclass
from typing import List, Optional, Dict, Any

@dataclass(frozen=True)
class StrategyConfig:
    stop_mode: str                 # "fixed" | "structural" | "bos"
    fixed_sl_pct: float
    buffer_pct: float
    tp_mode: TakeProfitMode
    tp_mult: float
    same_bar_rule: SameBarSlTpRule

def build_artem_configs() -> List[StrategyConfig]:
    stop_modes = ["fixed", "structural"]
    tp_modes   = [TakeProfitMode.RR_BASED, TakeProfitMode.RANGE_BASED]
    fixed_sls  = [0.005, 0.01]
    buffers    = [0.0, 0.001, 0.002]
    tp_mults   = [1.0, 1.5, 2.0]
    same_bar   = [SameBarSlTpRule.WORST_CASE]

    out = []
    for sm in stop_modes:
        for tp in tp_modes:
            for sl in fixed_sls:
                for buf in buffers:
                    for mult in tp_mults:
                        for sbr in same_bar:
                            out.append(StrategyConfig(sm, float(sl), float(buf), tp, float(mult), sbr))
    return out

def build_david_configs() -> List[StrategyConfig]:
    stop_modes = ["structural", "bos"]
    tp_modes   = [TakeProfitMode.RR_BASED]
    fixed_sls  = [0.005]
    buffers    = [0.001, 0.002]
    tp_mults   = [1.5, 2.0]
    same_bar   = [SameBarSlTpRule.WORST_CASE, SameBarSlTpRule.OPEN_PROXIMITY]

    out = []
    for sm in stop_modes:
        for tp in tp_modes:
            for sl in fixed_sls:
                for buf in buffers:
                    for mult in tp_mults:
                        for sbr in same_bar:
                            out.append(StrategyConfig(sm, float(sl), float(buf), tp, float(mult), sbr))
    return out

artem_configs = build_artem_configs()
david_configs = build_david_configs()
print("configs:", "artem", len(artem_configs), "| david", len(david_configs))

In [4]:
# --- Fast backtest engine (cache bars, avoid QC .NET Bar collisions)

def _prep_bars_and_swings(df: pd.DataFrame, swings: pd.DataFrame):
    BarCls = ee.Bar  # critical: use python Bar from entry_exit

    bars = [
        BarCls(
            time=idx,
            open=float(row["open"]),
            high=float(row["high"]),
            low=float(row["low"]),
            close=float(row["close"]),
            volume=float(row["volume"]) if "volume" in row else None,
        )
        for idx, row in df.iterrows()
    ]

    n = len(df)
    hl = np.full(n, np.nan, dtype=float)
    lv = np.full(n, np.nan, dtype=float)
    if swings is not None and len(swings) > 0:
        m = min(n, len(swings))
        if "HighLow" in swings.columns:
            hl[:m] = swings.iloc[:m]["HighLow"].astype(float).values
        if "Level" in swings.columns:
            lv[:m] = swings.iloc[:m]["Level"].astype(float).values

    return bars, hl, lv


def backtest_bos_breakout(
    df: pd.DataFrame,
    swings: pd.DataFrame,
    risk_cfg: RiskConfig,
    cfg: StrategyConfig,
    *,
    initial_cash: float = 100000.0,
    buying_power_cash: Optional[float] = None,
    collect_curve: bool = True,
) -> Dict[str, Any]:

    bars, hl_arr, lv_arr = _prep_bars_and_swings(df, swings)

    swing_levels = ee.SwingLevels()
    equity = float(initial_cash)
    peak = equity
    max_dd = 0.0

    in_pos = False
    plan = None

    trades = 0
    wins = 0
    losses = 0

    equity_curve = [equity] if collect_curve else None

    for t, bar in enumerate(bars):

        # swing update (NEW API)
        hl = hl_arr[t]
        lvl = lv_arr[t]
        if not np.isnan(hl) and not np.isnan(lvl):
            swing_levels = ee.update_last_swing_levels(
                swing_levels=swing_levels,
                highlow_flag=float(hl),
                level=float(lvl),
            )

        # exit
        if in_pos and plan is not None:
            exit_ev = ee.check_exit_rules(
                bar=bar,
                direction=plan.direction,
                sl_price=float(plan.sl_price),
                tp_price=(float(plan.tp_price) if plan.tp_price is not None else None),
                same_bar_rule=cfg.same_bar_rule,
            )
            if exit_ev is not None:
                entry_price = float(plan.entry_price)
                exit_price  = float(exit_ev.exit_price)
                qty         = float(plan.quantity)

                pnl = (exit_price - entry_price) * qty if plan.direction == ee.PositionDirection.LONG else (entry_price - exit_price) * qty
                equity += pnl

                trades += 1
                if pnl >= 0:
                    wins += 1
                else:
                    losses += 1

                in_pos = False
                plan = None

        # entry
        if (not in_pos) and (t < len(bars) - 1):
            sig = ee.detect_bos_signal(bars=bars, t=t, swing_levels=swing_levels)
            if sig is not None:
                slm = StopLossManager(mode=cfg.stop_mode, fixed_pct=cfg.fixed_sl_pct, buffer_pct=cfg.buffer_pct)

                try:
                    plan = ee.plan_trade_from_signal(
                        bars=bars,
                        bos_signal=sig,
                        swing_levels=swing_levels,
                        stop_loss_manager=slm,
                        tp_mode=cfg.tp_mode,
                        tp_mult=cfg.tp_mult,
                        risk_config=risk_cfg,
                        buying_power_cash=buying_power_cash,
                    )
                except ValueError as e:
                    # sizing refused or invalid TP range etc. -> skip entry
                    plan = None

                if plan is not None and plan.quantity is not None and float(plan.quantity) > 0:
                    in_pos = True
                else:
                    plan = None


        # drawdown
        if equity > peak:
            peak = equity
        dd = (peak - equity) / peak if peak > 0 else 0.0
        max_dd = max(max_dd, dd)

        if collect_curve:
            equity_curve.append(equity)

    metrics = {
        "final_equity": float(equity),
        "trades": int(trades),
        "wins": int(wins),
        "losses": int(losses),
        "win_rate": float(wins / trades) if trades > 0 else 0.0,
        "max_drawdown": float(max_dd),
    }
    return {"metrics": metrics, "equity_curve": equity_curve}


def grid_search(
    df: pd.DataFrame,
    swings: pd.DataFrame,
    risk_cfg: RiskConfig,
    configs: List[StrategyConfig],
    *,
    objective: str = "final_equity",
    initial_cash: float = 100000.0,
    buying_power_cash: Optional[float] = None,
    skip_value_errors: bool = True,
) -> pd.DataFrame:
    rows = []

    for cfg in configs:
        try:
            res = backtest_bos_breakout(
                df, swings, risk_cfg, cfg,
                initial_cash=initial_cash,
                buying_power_cash=buying_power_cash,
                collect_curve=False,
            )
            m = res["metrics"]

            rows.append({
                "objective": float(m.get(objective, np.nan)),
                "trades": int(m.get("trades", 0)),
                "win_rate": float(m.get("win_rate", 0.0)),
                "max_drawdown": float(m.get("max_drawdown", 0.0)),
                "stop_mode": cfg.stop_mode.value if hasattr(cfg.stop_mode, "value") else str(cfg.stop_mode),
                "fixed_sl_pct": float(cfg.fixed_sl_pct),
                "buffer_pct": float(cfg.buffer_pct),
                "tp_mode": cfg.tp_mode.value if hasattr(cfg.tp_mode, "value") else str(cfg.tp_mode),
                "tp_mult": float(cfg.tp_mult),
                "same_bar_rule": cfg.same_bar_rule.value if hasattr(cfg.same_bar_rule, "value") else str(cfg.same_bar_rule),
                "skipped": False,
                "skip_reason": "",
            })

        except ValueError as e:
            if not skip_value_errors:
                raise
            # keep a record (optional but useful)
            rows.append({
                "objective": np.nan,
                "trades": 0,
                "win_rate": 0.0,
                "max_drawdown": 0.0,
                "stop_mode": cfg.stop_mode.value if hasattr(cfg.stop_mode, "value") else str(cfg.stop_mode),
                "fixed_sl_pct": float(cfg.fixed_sl_pct),
                "buffer_pct": float(cfg.buffer_pct),
                "tp_mode": cfg.tp_mode.value if hasattr(cfg.tp_mode, "value") else str(cfg.tp_mode),
                "tp_mult": float(cfg.tp_mult),
                "same_bar_rule": cfg.same_bar_rule.value if hasattr(cfg.same_bar_rule, "value") else str(cfg.same_bar_rule),
                "skipped": True,
                "skip_reason": str(e),
            })

    out = pd.DataFrame(rows)
    if len(out) == 0:
        return out

    # Prefer non-skipped rows; then objective desc, then trades desc
    if "skipped" in out.columns:
        out = out.sort_values(["skipped", "objective", "trades"], ascending=[True, False, False])
    else:
        out = out.sort_values(["objective", "trades"], ascending=[False, False])

    return out.reset_index(drop=True)



def time_split_holdout(
    df: pd.DataFrame,
    swings: pd.DataFrame,
    risk_cfg: RiskConfig,
    configs: List[StrategyConfig],
    *,
    split_frac: float = 0.7,
    objective: str = "final_equity",
    initial_cash: float = 100000.0,
    buying_power_cash: Optional[float] = None,
) -> Dict[str, Any]:

    cut = int(len(df) * split_frac)
    df_train, df_test = df.iloc[:cut], df.iloc[cut:]
    sw_train, sw_test = swings.iloc[:cut], swings.iloc[cut:]

    gs = grid_search(df_train, sw_train, risk_cfg, configs,
                     objective=objective, initial_cash=initial_cash, buying_power_cash=buying_power_cash)
    if len(gs) == 0:
        return {"grid": gs, "best_cfg": None, "is_metrics": {}, "oos_metrics": {}}

    best = gs.iloc[0]
    best_cfg = StrategyConfig(
        stop_mode=str(best["stop_mode"]),
        fixed_sl_pct=float(best["fixed_sl_pct"]),
        buffer_pct=float(best["buffer_pct"]),
        tp_mode=TakeProfitMode(best["tp_mode"]),
        tp_mult=float(best["tp_mult"]),
        same_bar_rule=SameBarSlTpRule(best["same_bar_rule"]),
    )

    is_res  = backtest_bos_breakout(df_train, sw_train, risk_cfg, best_cfg, initial_cash=initial_cash, buying_power_cash=buying_power_cash, collect_curve=True)
    oos_res = backtest_bos_breakout(df_test,  sw_test,  risk_cfg, best_cfg, initial_cash=initial_cash, buying_power_cash=buying_power_cash, collect_curve=True)

    return {
        "grid": gs,
        "best_cfg": best_cfg,
        "is_metrics": is_res["metrics"],
        "oos_metrics": oos_res["metrics"],
        "is_curve": is_res["equity_curve"],
        "oos_curve": oos_res["equity_curve"],
    }


def walk_forward_analysis(
    df: pd.DataFrame,
    swings: pd.DataFrame,
    risk_cfg: RiskConfig,
    configs: List[StrategyConfig],
    *,
    train_bars: int = 20000,
    test_bars: int = 5000,
    anchored: bool = True,
    objective: str = "final_equity",
    initial_cash: float = 100000.0,
    buying_power_cash: Optional[float] = None,
) -> pd.DataFrame:

    rows = []
    n = len(df)
    start = 0
    while True:
        train_start = 0 if anchored else start
        train_end = train_start + train_bars
        test_end = train_end + test_bars
        if test_end > n:
            break

        df_train = df.iloc[train_start:train_end]
        sw_train = swings.iloc[train_start:train_end]
        df_test  = df.iloc[train_end:test_end]
        sw_test  = swings.iloc[train_end:test_end]

        gs = grid_search(df_train, sw_train, risk_cfg, configs,
                         objective=objective, initial_cash=initial_cash, buying_power_cash=buying_power_cash)
        if len(gs) == 0:
            break

        best = gs.iloc[0]
        best_cfg = StrategyConfig(
            stop_mode=str(best["stop_mode"]),
            fixed_sl_pct=float(best["fixed_sl_pct"]),
            buffer_pct=float(best["buffer_pct"]),
            tp_mode=TakeProfitMode(best["tp_mode"]),
            tp_mult=float(best["tp_mult"]),
            same_bar_rule=SameBarSlTpRule(best["same_bar_rule"]),
        )

        oos = backtest_bos_breakout(df_test, sw_test, risk_cfg, best_cfg,
                                    initial_cash=initial_cash, buying_power_cash=buying_power_cash, collect_curve=False)["metrics"]

        rows.append({
            "train_start": df_train.index.min(),
            "train_end": df_train.index.max(),
            "test_start": df_test.index.min(),
            "test_end": df_test.index.max(),
            "best_stop_mode": best_cfg.stop_mode,
            "best_tp_mode": best_cfg.tp_mode.value,
            "best_tp_mult": best_cfg.tp_mult,
            "best_same_bar_rule": best_cfg.same_bar_rule.value,
            **{f"oos_{k}": v for k, v in oos.items()},
        })

        start += test_bars

    return pd.DataFrame(rows)

In [5]:
# --- Run: BTC/ETH + baselines + swaps

risk_per_trade = 0.01
max_leverage   = 1.0

initial_cash = 100000.0
total_value  = initial_cash

risk_cfg = RiskConfig(
    risk_budget_cash=total_value * risk_per_trade,
    use_buying_power_cap=True,
)
buying_power_cash = total_value * max_leverage

artem_symbol = "BTCUSD"
artem_start  = datetime(2024, 1, 1)
artem_end    = datetime(2025, 1, 1)

david_symbol = "ETHUSD"
david_start  = datetime(2024, 1, 1)
david_end    = datetime(2025, 1, 1)

df_btc = load_crypto_history(artem_symbol, artem_start, artem_end, Resolution.Minute)
df_eth = load_crypto_history(david_symbol, david_start, david_end, Resolution.Minute)

df_btc_15 = resample_ohlc(df_btc, "15min")
df_eth_15 = resample_ohlc(df_eth, "15min")

sw_btc = compute_swings(df_btc_15)
sw_eth = compute_swings(df_eth_15)

print("Data loaded:")
print(" - BTC:", df_btc_15.shape, df_btc_15.index.min(), "->", df_btc_15.index.max())
print(" - ETH:", df_eth_15.shape, df_eth_15.index.min(), "->", df_eth_15.index.max())
print("Risk:", f"initial_cash={initial_cash:.2f}", f"risk_budget_cash={risk_cfg.risk_budget_cash:.2f}", f"buying_power_cash={buying_power_cash:.2f}")

baseline_artem_btc = time_split_holdout(df_btc_15, sw_btc, risk_cfg, artem_configs, split_frac=0.7,
                                       initial_cash=initial_cash, buying_power_cash=buying_power_cash)
baseline_david_eth = time_split_holdout(df_eth_15, sw_eth, risk_cfg, david_configs, split_frac=0.7,
                                       initial_cash=initial_cash, buying_power_cash=buying_power_cash)

swap_artem_on_eth = time_split_holdout(df_eth_15, sw_eth, risk_cfg, artem_configs, split_frac=0.7,
                                      initial_cash=initial_cash, buying_power_cash=buying_power_cash)
swap_david_on_btc = time_split_holdout(df_btc_15, sw_btc, risk_cfg, david_configs, split_frac=0.7,
                                      initial_cash=initial_cash, buying_power_cash=buying_power_cash)

summary = pd.DataFrame([
    {"experiment":"baseline_artem (artem configs on BTC)", **baseline_artem_btc["oos_metrics"]},
    {"experiment":"baseline_david (david configs on ETH)", **baseline_david_eth["oos_metrics"]},
    {"experiment":"swap_artem_on_eth (artem configs on ETH)", **swap_artem_on_eth["oos_metrics"]},
    {"experiment":"swap_david_on_btc (david configs on BTC)", **swap_david_on_btc["oos_metrics"]},
])
summary

In [ ]:
# --- Optional: walk-forward (can take time but should not crash kernel)

wf_btc_artem = walk_forward_analysis(df_btc_15, sw_btc, risk_cfg, artem_configs,
                                    train_bars=20000, test_bars=5000, anchored=True,
                                    initial_cash=initial_cash, buying_power_cash=buying_power_cash)

wf_eth_david = walk_forward_analysis(df_eth_15, sw_eth, risk_cfg, david_configs,
                                    train_bars=20000, test_bars=5000, anchored=True,
                                    initial_cash=initial_cash, buying_power_cash=buying_power_cash)

print("WF BTC/Artem rows:", len(wf_btc_artem))
print("WF ETH/David rows:", len(wf_eth_david))

wf_btc_artem.head()